In [ ]:
import pandas as pd

# =========================================================
# 1. read file
# =========================================================
price = pd.read_excel(
    r"C:\Users\Dell\Desktop\Project\Price.xlsx"
)

weather = pd.read_excel(
    r"C:\Users\Dell\Desktop\Project\Weather.xlsx"
)

# =========================================================
# 2. convert date format
# =========================================================
price["Start date"] = pd.to_datetime(
    price["Start date"],
    errors="coerce", format="%Y-%m-%d %H:%M:%S"
)

price["End date"] = pd.to_datetime(
    price["End date"],
    errors="coerce", format="%Y-%m-%d %H:%M:%S"
)

weather["date"] = pd.to_datetime(
    weather["date"],
    errors="coerce"
)

print("Price rows:", len(price))
print("Weather rows:", len(weather))

# =========================================================
# 3. select date range
# =========================================================
weather = weather[
    (weather["date"] >= "2025-01-01") &
    (weather["date"] < "2026-09-30")
].copy()

price = price[
    (price["Start date"] >= "2025-01-01") &
    (price["Start date"] < "2026-09-30")
].copy()

print("filtered Weather rows:", len(weather))
print("Weather earliest date:", weather["date"].min())
print("Weather latest date:", weather["date"].max())

print("filtered Price rows:", len(price))
print("Price earliest date:", price["Start date"].min())
print("Price latest date:", price["Start date"].max())

# =========================================================
# 4. eliminate unrelated price list
# =========================================================
price = price[
    [
        "Start date",
        "Germany/Luxembourg [€/MWh]",
        "∅ DE/LU neighbours [€/MWh]"
    ]
].copy()

print(price.columns.tolist())
print("Price rows:", len(price))
print("Weather rows:", len(weather))

# =========================================================
# 5. check 2025-10-26 unusual timezone convertion
# =========================================================
print("\n--- Weather around 2025-10-26 ---")
print(
    weather[
        (weather["date"] >= "2025-10-26 00:00") &
        (weather["date"] <= "2025-10-26 04:00")
    ]["date"].tolist()
)

print("\n--- Price around 2025-10-26 ---")
print(
    price[
        (price["Start date"] >= "2025-10-26 00:00") &
        (price["Start date"] <= "2025-10-26 04:00")
    ]["Start date"].tolist()
)

# =========================================================
# 6. assign time order for matching（ 1-to-1 ）
# =========================================================
price_merge = price.copy()
weather_merge = weather.copy()

price_merge["time_order"] = (
    price_merge.groupby("Start date").cumcount()
)

weather_merge["time_order"] = (
    weather_merge.groupby("date").cumcount()
)

# =========================================================
# 7. merge
# =========================================================
merged = pd.merge(
    price_merge,
    weather_merge,
    left_on=["Start date", "time_order"],
    right_on=["date", "time_order"],
    how="inner"
)

merged = merged.reset_index(drop=True)

# delete duplicate dates column
merged = merged.drop(columns=["date", "time_order"])

print("\nMerged rows:", len(merged))
print(merged.head())

# =========================================================
# 8. export file after merge
# =========================================================
merged.to_excel(
    r"C:\Users\Dell\Desktop\Project\Merged.xlsx",
    index=False
)

print("\nSaved to Merged.xlsx")